# SalesLens - Data Exploration

## 1. Project Objective
We inspect the raw Olist CSV files before any cleaning so we can understand their structure, quality, keys, dates, and relationships.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from src.raw_loader import list_raw_csv_files, load_raw_csv

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 2. Load Raw Data

In [ ]:
files = list_raw_csv_files()
files

## 3. Dataset Overview

In [ ]:
data = {p.name: load_raw_csv(p.name) for p in files}
overview = pd.DataFrame([
    {
        'Dataset': name,
        'Nombre de lignes': len(df),
        'Nombre de colonnes': len(df.columns),
        'Valeurs manquantes': int(df.isna().sum().sum()),
        'Doublons': int(df.duplicated().sum()),
    }
    for name, df in data.items()
]).sort_values('Dataset')
overview

## 4. Missing Values

In [ ]:
for name, df in data.items():
    print(f'\n### {name}')
    print(df.isna().sum().sort_values(ascending=False))


## 5. Duplicate Records

In [ ]:
for name, df in data.items():
    print(f'{name}: {df.duplicated().sum()} duplicates')


## 6. Identifiers and Relationships

In [ ]:
for name, df in data.items():
    print(f'\n### {name}')
    id_cols = [c for c in df.columns if c.endswith('_id') or c.endswith('_code_prefix') or c in {'review_id', 'order_item_id', 'payment_sequential'}]
    for c in id_cols:
        print(f'{c}: unique={df[c].nunique(dropna=False)} / rows={len(df)}')


## 7. Date Columns

In [ ]:
date_candidates = {
    'olist_order_items_dataset.csv': ['shipping_limit_date'],
    'olist_order_reviews_dataset.csv': ['review_creation_date', 'review_answer_timestamp'],
    'olist_orders_dataset.csv': [
        'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_carrier_date', 'order_delivered_customer_date',
        'order_estimated_delivery_date']
}
for name, cols in date_candidates.items():
    df = data[name]
    print(f'\n### {name}')
    for c in cols:
        parsed = pd.to_datetime(df[c], errors='coerce')
        print(c, '| dtype=', df[c].dtype, '| parsed_dtype=', parsed.dtype, '| min=', parsed.min(), '| max=', parsed.max())


## 8. Numerical Variables

In [ ]:
numeric_cols = {
    'olist_order_items_dataset.csv': ['price', 'freight_value'],
    'olist_order_payments_dataset.csv': ['payment_installments', 'payment_value'],
    'olist_order_reviews_dataset.csv': ['review_score'],
    'olist_products_dataset.csv': ['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'],
}
for name, cols in numeric_cols.items():
    df = data[name]
    print(f'\n### {name}')
    print(df[cols].describe().T[['count', 'mean', 'min', '50%', 'max']])


## 9. Categorical Variables

In [ ]:
cat_cols = {
    'olist_orders_dataset.csv': ['order_status'],
    'olist_order_payments_dataset.csv': ['payment_type'],
    'olist_customers_dataset.csv': ['customer_state'],
    'olist_sellers_dataset.csv': ['seller_state'],
    'olist_products_dataset.csv': ['product_category_name'],
}
for name, cols in cat_cols.items():
    df = data[name]
    print(f'\n### {name}')
    for c in cols:
        print(f'\n{c}')
        print(df[c].value_counts(dropna=False).head(10))


## 10. Initial Observations
- The data covers the 2016-2018 Olist marketplace period.
- Orders, customers, products, sellers, payments, reviews, and geolocation data are split across separate tables.
- `order_items` is the most granular transactional table, so one order can appear multiple times there.
- `geolocation` has many duplicate rows, which suggests repeated postal-code/location combinations rather than a clean unique list.
- `order_reviews` contains a lot of missing comment fields, which is expected for optional text feedback.
- Several date columns are stored as text and will need conversion later, but not yet in this step.